# Home Tour Route Optimizer

**Problem:** Given N candidate homes with open house time windows, find the subset and visit ordering that maximizes total interest score within a time budget.

This notebook walks through the full pipeline:
1. Load and preprocess home listing data
2. Score homes based on buyer preferences
3. Geocode addresses and build a travel time matrix
4. Solve the Orienteering Problem with Time Windows (OPTW)

# Problem Framing

Given a set of open houses around the Lawrenceville/Lilburn/Duluth area (GA) with listing prices between \$350K–\$500K on a Saturday, we want to plan an optimal tour. Each home has:
- An **interest score** reflecting buyer preferences
- An **open house time window** (e.g., 13:00–15:00)
- A **visit duration** (time spent at each showing)

The goal is to **select a subset of homes and determine the visit order** that maximizes total interest score while respecting time windows and a total tour budget.

This is a variant of the **Orienteering Problem with Time Windows (OPTW)** — a classic NP-hard combinatorial optimization problem that blends elements of the Knapsack Problem (subset selection) and the Traveling Salesman Problem (routing).

# Data

Listings were sampled from Redfin to reflect a realistic home search scenario. The dataset includes 17 homes with property details, school information, neighborhood scores, and open house windows.

In [1]:
import pandas as pd
df = pd.read_csv('../data/demo_homes.csv')
print(f"Loaded {len(df)} homes with {df.shape[1]} columns")
df[['address','price','beds','baths','sqft','year_built','hoa_fee','garage_spaces']]

# df.to_csv('data/demo_homes.csv', index=False)

Loaded 17 homes with 29 columns


,address,price,beds,baths,sqft,year_built,hoa_fee,garage_spaces
0,"2743 Hallwood Ln, Suwanee, GA 30024",499900,4,3.5,2539,2026,250,2
1,"5652 Plain Field Ln, Lilburn, GA 30047",380000,3,2.5,1597,2021,310,1
2,"3487 Sorrel Ln, Duluth, GA 30096",469999,3,2.5,2225,2020,180,2
3,"310 Melanie Way, Lawrenceville, GA 30044",439000,3,2.5,3870,1988,0,2
4,"1208 Gate Post Ln, Lawrenceville, GA 30044",360000,3,2.5,1632,1985,0,2
5,"255 Timber Oak Cv, Lawrenceville, GA 30043",435000,4,2.5,2537,1987,0,2
6,"2583 Madison Mae Ln, Grayson, GA 30017",449900,4,3.0,2307,2017,109,2
7,"3109 Misty Springs Path, Lawrenceville, GA 30045",402990,4,2.5,2143,2026,130,2
8,"1386 Image Xing, Lawrenceville, GA 30045",418500,3,2.5,2253,2017,125,2
9,"100 Epperly Way, Tucker, GA 30084",456990,3,3.5,1893,2026,125,2


## Data Preprocessing

Handle missing values in the neighborhood score columns before scoring.

In [2]:
missing_pct = df.isna().mean() * 100
result = missing_pct[missing_pct > 0]
print(result)

bike_score         5.882353
quiet_score       11.764706
wellness_score    17.647059
vibrancy_score    11.764706
dtype: float64


In [3]:
# Temporary forward-fill placeholder so further code doesn't crash. Eventually improve method

df['bike_score']      = df['bike_score'].fillna(df['bike_score'].median())
df['quiet_score']     = df['quiet_score'].fillna(df['quiet_score'].median())
df['wellness_score']  = df['wellness_score'].fillna(df['wellness_score'].median())
df['vibrancy_score']  = df['vibrancy_score'].fillna(df['vibrancy_score'].median())

# Interest Score

Each home receives an interest score (0–100) based on buyer preferences using a **weighted linear scoring model** — a standard approach in multi-criteria decision making (MCDM).

The scoring pipeline:
1. **Dealbreaker filter** — hard constraints (min beds, max price, etc.) eliminate non-starters
2. **Weighted features** — price, sqft, beds/baths, year built, lot size, garage, HOA, school ratings
3. **Neighborhood composite** — quiet, walkable, bike-friendly, wellness, vibrancy scores with a preferred-vibe multiplier
4. **Property type penalty** — non-preferred types receive a small score reduction

The interactive widget below collects preferences and produces the scores.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Layout constants ─────────────────────────────────────────────────────────
W_FULL   = widgets.Layout(width='100%')
W_SLIDER = widgets.Layout(width='380px')
W_WIDE   = widgets.Layout(width='500px')
W_NUM_SM = widgets.Layout(width='200px')
W_NUM_MD = widgets.Layout(width='230px')

SECTION_BOX = dict(
    padding='16px 20px',
    margin='0 0 12px 0',
    border='1px solid #e0e0e0',
    border_radius='10px',
    width='580px',
)

def section(title, subtitle, *children):
    """Titled, bordered card wrapping a group of widgets."""
    header = widgets.VBox([
        widgets.HTML(
            f'<div style="font-size:15px;font-weight:600;color:#1a1a1a;margin-bottom:2px">{title}</div>'
            f'<div style="font-size:12px;color:#888;line-height:1.4">{subtitle}</div>'
        )
    ], layout=widgets.Layout(margin='0 0 12px 0'))
    return widgets.VBox(
        [header, *children],
        layout=widgets.Layout(**SECTION_BOX)
    )

def weight_row(label, sublabel, key, default):
    """
    A single importance slider row with:
      - bold label + muted sublabel stacked on the left
      - slider in the middle
      - 'Not important … Essential' anchor labels on the ends
    """
    left = widgets.HTML(
        f'<div style="min-width:190px;max-width:190px">'
        f'<div style="font-size:13px;font-weight:500;color:#1a1a1a">{label}</div>'
        f'<div style="font-size:11px;color:#999;margin-top:1px">{sublabel}</div>'
        f'</div>'
    )
    sl = widgets.IntSlider(
        value=default, min=0, max=10, step=1,
        continuous_update=True,
        layout=W_SLIDER,
        style={'description_width': '0px'},
    )
    anchors = widgets.HTML(
        '<div style="display:flex;justify-content:space-between;'
        'width:380px;font-size:10px;color:#aaa;margin-top:2px">'
        '<span>Doesn\'t matter</span><span>Essential</span></div>'
    )
    row = widgets.VBox([
        widgets.HBox([left, sl]),
        widgets.HBox([widgets.HTML('<div style="min-width:190px"></div>'), anchors]),
    ], layout=widgets.Layout(margin='0 0 10px 0'))
    return row, key, sl


# ════════════════════════════════════════════════════════════════════════════
# SECTION 1 · Must-haves
# Hard filters — homes that fail any of these are removed before scoring.
# ════════════════════════════════════════════════════════════════════════════

db_no_hoa = widgets.Checkbox(
    value=False,
    description='I will not consider homes with an HOA fee',
    layout=W_FULL,
    style={'description_width': 'initial'},
)

db_min_beds = widgets.BoundedIntText(
    value=0, min=0, max=10,
    description='Bedrooms (min):',
    layout=W_NUM_SM,
    style={'description_width': '120px'},
)
db_min_baths = widgets.BoundedFloatText(
    value=0.0, min=0, max=10, step=0.5,
    description='Bathrooms (min):',
    layout=W_NUM_SM,
    style={'description_width': '120px'},
)
db_min_sqft = widgets.BoundedIntText(
    value=0, min=0, max=10000, step=100,
    description='Square feet (min):',
    layout=W_NUM_MD,
    style={'description_width': '130px'},
)
db_max_price = widgets.BoundedIntText(
    value=600000, min=0, max=5000000, step=10000,
    description='Budget max ($):',
    layout=W_NUM_MD,
    style={'description_width': '130px'},
)
db_min_garage = widgets.BoundedIntText(
    value=0, min=0, max=5,
    description='Garage spaces (min):',
    layout=W_NUM_MD,
    style={'description_width': '145px'},
)
db_min_year = widgets.BoundedIntText(
    value=0, min=0, max=2026, step=1,
    description='Built after year:',
    layout=W_NUM_MD,
    style={'description_width': '110px'},
)
db_note = widgets.HTML(
    '<div style="font-size:11px;color:#aaa;margin-top:4px">'
    'Leave any field at 0 to skip that requirement.</div>'
)

sec_musthaves = section(
    '🚫 Must-haves',
    'Any home that doesn\'t meet these requirements will be removed before ranking.',
    db_no_hoa,
    widgets.HBox([db_min_beds,  db_min_baths],  layout=widgets.Layout(margin='8px 0 0')),
    widgets.HBox([db_min_sqft,  db_max_price],  layout=widgets.Layout(margin='6px 0 0')),
    widgets.HBox([db_min_garage, db_min_year],  layout=widgets.Layout(margin='6px 0 0')),
    db_note,
)


# ════════════════════════════════════════════════════════════════════════════
# SECTION 2 · What matters most to you?
# Importance sliders — each maps to a feature column in the homes dataset.
# ════════════════════════════════════════════════════════════════════════════

weight_configs = [
    # label            sublabel                              key           default
    ('Price',          'Lower price = higher score',         'price',         7),
    ('Size of home',   'Total square footage',               'sqft',          6),
    ('Beds & baths',   'Number of bedrooms and bathrooms',   'beds_baths',    7),
    ('How new it is',  'Year the home was built',            'year_built',    5),
    ('Outdoor space',  'Lot size and yard',                  'lot_size',      4),
    ('Parking',        'Number of garage spaces',            'garage',        4),
    ('HOA fee',        'Lower monthly fee = higher score',   'hoa_fee',       3),
    ('Elementary school', 'GreatSchools rating (1–10)',      'elem_rating',   5),
    ('Middle school',     'GreatSchools rating (1–10)',      'mid_rating',    5),
    ('High school',       'GreatSchools rating (1–10)',      'high_rating',   5),
]

weight_rows_widgets = []
weight_sliders = {}
for label, sublabel, key, default in weight_configs:
    row, k, sl = weight_row(label, sublabel, key, default)
    weight_rows_widgets.append(row)
    weight_sliders[k] = sl

sec_weights = section(
    '⭐ What matters most to you?',
    'Slide each one to show how important that feature is. '
    'Set it to 0 if you don\'t care about it at all.',
    *weight_rows_widgets,
)


# ════════════════════════════════════════════════════════════════════════════
# SECTION 3 · Neighborhood feel
# Maps to: quiet_score, walk_score, bike_score, wellness_score, vibrancy_score
# Preferred vibe column gets a 2× multiplier in scoring.
# ════════════════════════════════════════════════════════════════════════════

vibe_selector = widgets.RadioButtons(
    options=[
        ('Quiet and peaceful — low traffic, residential feel',  'quiet'),
        ('Walkable — close to shops, restaurants, errands',     'walk'),
        ('Bike-friendly — good paths and low-traffic streets',  'bike'),
        ('Health & wellness — gyms, parks, healthy dining',     'wellness'),
        ('Lively and vibrant — lots of activity and energy',    'vibrancy'),
        ('No preference — treat all equally',                   'all'),
    ],
    value='all',
    layout=W_FULL,
    style={'description_width': 'initial'},
)
vibe_weight_slider = widgets.IntSlider(
    value=6, min=0, max=10, step=1,
    continuous_update=True,
    layout=W_SLIDER,
    style={'description_width': '0px'},
)
vibe_weight_label = widgets.HTML(
    '<div style="font-size:13px;font-weight:500;color:#1a1a1a;'
    'min-width:190px">How much does neighborhood feel matter overall?</div>'
)
vibe_anchors = widgets.HTML(
    '<div style="display:flex;justify-content:space-between;'
    'width:380px;font-size:10px;color:#aaa;margin-top:2px">'
    '<span>Doesn\'t matter</span><span>Essential</span></div>'
)

sec_vibe = section(
    '🏙️ Neighborhood feel',
    'Pick the vibe that matters most to you — that factor will be weighted more heavily in the ranking.',
    vibe_selector,
    widgets.HTML('<div style="margin-top:10px"></div>'),
    widgets.HBox([vibe_weight_label, vibe_weight_slider]),
    widgets.HBox([widgets.HTML('<div style="min-width:190px"></div>'), vibe_anchors]),
)


# ════════════════════════════════════════════════════════════════════════════
# SECTION 4 · Property type
# Non-matching homes receive a –10 point penalty (not eliminated).
# ════════════════════════════════════════════════════════════════════════════

prop_type_checks = {
    'Townhome':     widgets.Checkbox(value=False, description='Townhome',     style={'description_width':'initial'}),
    'Single Family':widgets.Checkbox(value=False, description='Single Family',style={'description_width':'initial'}),
    'Condo':        widgets.Checkbox(value=False, description='Condo',        style={'description_width':'initial'}),
}
prop_no_pref = widgets.Checkbox(
    value=True, description='No preference',
    style={'description_width': 'initial'}
)

# When "No preference" is checked, uncheck the others and disable them
def _on_no_pref(change):
    if change['new']:
        for cb in prop_type_checks.values():
            cb.value = False
            cb.disabled = True
    else:
        for cb in prop_type_checks.values():
            cb.disabled = False

def _on_type_check(change):
    if change['new']:
        prop_no_pref.value = False

prop_no_pref.observe(_on_no_pref, names='value')
for cb in prop_type_checks.values():
    cb.observe(_on_type_check, names='value')

# Trigger initial state
_on_no_pref({'new': True})

prop_note = widgets.HTML(
    '<div style="font-size:11px;color:#aaa;margin-top:4px">'
    'Homes that don\'t match your selection are kept but ranked slightly lower.</div>'
)

sec_proptype = section(
    '🏠 Type of home',
    'What kind of property are you open to? Check all that apply.',
    widgets.HBox(list(prop_type_checks.values()) + [prop_no_pref],
                 layout=widgets.Layout(flex_wrap='wrap', gap='12px')),
    prop_note,
)


# ════════════════════════════════════════════════════════════════════════════
# SECTION 5 · Your day
# Touring schedule — passed to the OPTW solver, not used in scoring.
# ════════════════════════════════════════════════════════════════════════════

tour_start = widgets.Text(
    value='09:00', description='I can start at:',
    layout=widgets.Layout(width='230px'),
    style={'description_width': '110px'},
)
tour_end = widgets.Text(
    value='18:00', description='I need to finish by:',
    layout=widgets.Layout(width='250px'),
    style={'description_width': '130px'},
)
visit_duration = widgets.BoundedIntText(
    value=30, min=10, max=120, step=5,
    description='Minutes per visit:',
    layout=widgets.Layout(width='230px'),
    style={'description_width': '125px'},
)
travel_buffer = widgets.BoundedIntText(
    value=15, min=0, max=60, step=5,
    description='Travel time buffer:',
    layout=widgets.Layout(width='250px'),
    style={'description_width': '130px'},
)
schedule_note = widgets.HTML(
    '<div style="font-size:11px;color:#aaa;margin-top:4px">'
    'Travel time buffer is extra time added between stops to account for parking, '
    'traffic, etc.</div>'
)

sec_schedule = section(
    '🕐 Your day',
    'When are you available to tour? This is used to plan your route — not your ranking.',
    widgets.HBox([tour_start, tour_end],       layout=widgets.Layout(gap='16px')),
    widgets.HBox([visit_duration, travel_buffer], layout=widgets.Layout(gap='16px', margin='8px 0 0')),
    schedule_note,
)


# ════════════════════════════════════════════════════════════════════════════
# SECTION 6 · Ranking cutoff
# Homes below min_score are excluded from the route solver entirely.
# ════════════════════════════════════════════════════════════════════════════

min_score_slider = widgets.IntSlider(
    value=30, min=0, max=90, step=5,
    continuous_update=True,
    layout=W_SLIDER,
    style={'description_width': '0px'},
)
cutoff_anchors = widgets.HTML(
    '<div style="display:flex;justify-content:space-between;'
    'width:380px;font-size:10px;color:#aaa;margin-top:2px">'
    '<span>Include everything</span><span>Only my top picks</span></div>'
)
cutoff_label = widgets.HTML(
    '<div style="font-size:13px;font-weight:500;min-width:190px;color:#1a1a1a">'
    'How selective should the route be?</div>'
)
cutoff_note = widgets.HTML(
    '<div style="font-size:11px;color:#aaa;margin-top:6px">'
    'Homes that score below this threshold won\'t be included in your tour route, '
    'even if their open house fits your schedule.</div>'
)

sec_threshold = section(
    '🎯 How selective do you want to be?',
    'Use this to avoid wasting time on homes you\'re not excited about.',
    widgets.HBox([cutoff_label, min_score_slider]),
    widgets.HBox([widgets.HTML('<div style="min-width:190px"></div>'), cutoff_anchors]),
    cutoff_note,
)


# ════════════════════════════════════════════════════════════════════════════
# Submit + output
# ════════════════════════════════════════════════════════════════════════════

submit_btn = widgets.Button(
    description='Save my preferences',
    button_style='primary',
    layout=widgets.Layout(width='210px', height='38px'),
)
form_out = widgets.Output()

user_preferences = {}

def on_submit(_):
    global user_preferences

    # Property type: collect checked boxes; fall back to 'No preference'
    selected_types = [t for t, cb in prop_type_checks.items() if cb.value]
    if not selected_types:
        selected_types = ['No preference']

    user_preferences = {
        'dealbreakers': {
            'no_hoa':     db_no_hoa.value,
            'min_beds':   db_min_beds.value,
            'min_baths':  db_min_baths.value,
            'min_sqft':   db_min_sqft.value,
            'max_price':  db_max_price.value,
            'min_garage': db_min_garage.value,
            'min_year':   db_min_year.value,
        },
        'weights':         {k: sl.value for k, sl in weight_sliders.items()},
        'vibe':            vibe_selector.value,
        'vibe_weight':     vibe_weight_slider.value,
        'preferred_types': selected_types,
        'tour_start':      tour_start.value,
        'tour_end':        tour_end.value,
        'visit_duration':  visit_duration.value,
        'travel_buffer':   travel_buffer.value,
        'min_score':       min_score_slider.value,
    }
    with form_out:
        clear_output()
        import json
        print('✅ Preferences saved. Run the next cell to rank your homes.\n')
        print(json.dumps(user_preferences, indent=2))

submit_btn.on_click(on_submit)

form = widgets.VBox([
    widgets.HTML(
        '<div style="font-size:18px;font-weight:600;color:#1a1a1a;margin-bottom:4px">'
        'Home Tour Preferences</div>'
        '<div style="font-size:13px;color:#888;margin-bottom:16px">'
        'Fill this out once. Your answers are used to rank the homes and plan your tour route.</div>'
    ),
    sec_musthaves,
    sec_weights,
    sec_vibe,
    sec_proptype,
    sec_schedule,
    sec_threshold,
    widgets.HTML('<div style="margin-top:4px"></div>'),
    submit_btn,
    form_out,
], layout=widgets.Layout(max_width='620px', padding='8px'))

display(form)


In [6]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import os
sys.path.append(os.path.abspath('..'))  # go up one level to project root
from core.scoring import score_homes

# ── Run ──────────────────────────────────────────────────────────────────────
if not user_preferences:
    raise RuntimeError("user_preferences is empty — submit the form in Cell 2 first.")

ranked_df = score_homes(df, user_preferences)

print("\n── Ranked homes ─────────────────────────────────────────────────────────")
display(
    ranked_df[['address','interest_score','price','beds','baths','sqft',
               'year_built','hoa_fee','garage_spaces','property_type']]
    .style.background_gradient(subset=['interest_score'], cmap='RdYlGn')
    .format({'price': '${:,.0f}', 'interest_score': '{:.1f}'})
)

# ── Filter to homes eligible for the OPTW solver ─────────────────────────────
min_score     = user_preferences['min_score']
solver_pool   = ranked_df[
    (ranked_df['interest_score'] >= min_score) &
    (ranked_df['open_house_start'].notna())
].copy()

# ── Build clean OPTW input DataFrame ─────────────────────────────────────────
optw_input_df = solver_pool[['address', 'interest_score', 'open_house_start', 'open_house_end']].copy()
optw_input_df['tour_time'] = user_preferences['visit_duration']
optw_input_df = optw_input_df[['address', 'interest_score', 'tour_time', 'open_house_start', 'open_house_end']].reset_index(drop=True)

print(f"\n── OPTW solver input (score ≥ {min_score}) ── {len(optw_input_df)} home(s) ──────")
display(optw_input_df)

Dealbreaker filter: 0 home(s) removed, 17 remaining.

── Ranked homes ─────────────────────────────────────────────────────────


,address,interest_score,price,beds,baths,sqft,year_built,hoa_fee,garage_spaces,property_type
16,"2964 Olivine Dr SE, Dacula, GA 30019",100.0,"$499,900",5,3.000000,3086,2007,71,3,Single-Family
15,"2017 Preserve Creek Way, Loganville, GA 30052",94.5,"$395,000",4,2.500000,2974,2004,56,2,Single-Family
12,"2201 Colonial Oak Way, Stone Mountain, GA 30087",91.3,"$405,000",4,2.500000,2572,1973,0,2,Single-Family
6,"2583 Madison Mae Ln, Grayson, GA 30017",87.2,"$449,900",4,3.000000,2307,2017,109,2,Single-Family
11,"104 Green Love Ln, Grayson, GA 30017",86.8,"$462,000",4,2.500000,2586,2024,50,2,Single-Family
10,"67 Wilder Ridge Way, Lawrenceville, GA 30044",84.1,"$410,510",3,3.500000,1967,2025,107,2,Townhome
7,"3109 Misty Springs Path, Lawrenceville, GA 30045",82.2,"$402,990",4,2.500000,2143,2026,130,2,Single-Family
0,"2743 Hallwood Ln, Suwanee, GA 30024",77.3,"$499,900",4,3.500000,2539,2026,250,2,Townhome
9,"100 Epperly Way, Tucker, GA 30084",74.2,"$456,990",3,3.500000,1893,2026,125,2,Townhome
3,"310 Melanie Way, Lawrenceville, GA 30044",69.0,"$439,000",3,2.500000,3870,1988,0,2,Single-Family



── OPTW solver input (score ≥ 80) ── 7 home(s) ──────


,address,interest_score,tour_time,open_house_start,open_house_end
0,"2964 Olivine Dr SE, Dacula, GA 30019",100.0,30,13:00,15:00
1,"2017 Preserve Creek Way, Loganville, GA 30052",94.5,30,12:00,14:00
2,"2201 Colonial Oak Way, Stone Mountain, GA 30087",91.3,30,11:00,13:00
3,"2583 Madison Mae Ln, Grayson, GA 30017",87.2,30,12:00,15:00
4,"104 Green Love Ln, Grayson, GA 30017",86.8,30,15:00,17:00
5,"67 Wilder Ridge Way, Lawrenceville, GA 30044",84.1,30,10:00,18:00
6,"3109 Misty Springs Path, Lawrenceville, GA 30045",82.2,30,12:00,17:00


# Travel Time Matrix

To route between homes we need pairwise travel durations. The pipeline:
1. **Geocode** each address using a three-tier fallback: US Census → Nominatim/OSM → trilateration from school locations
2. **Build a duration matrix** via OSRM (real road network), falling back to haversine estimates if OSRM is unavailable

## Add Start and End Location

The solver needs a depot — where your day starts and ends. This gets prepended/appended to the home list.

In [10]:
# Function to add start and end location to

def add_start_and_end(optw_input_df, start_address, end_address):
    """
    Prepends a start depot row and appends an end depot row to optw_input_df.

    Start row: interest_score=0, tour_time=0, window = [tour_start, tour_end]
    End row:   interest_score=0, tour_time=0, window = [tour_start, tour_end]

    Parameters
    ----------
    optw_input_df : DataFrame with columns address, interest_score, tour_time,
                    open_house_start, open_house_end
    start_address : str — where your day begins
    end_address   : str — where your day ends (can be same as start_address)

    Returns
    -------
    DataFrame with start row prepended and end row appended, index reset.
    """
    tour_start = user_preferences['tour_start']
    tour_end   = user_preferences['tour_end']

    start_row = pd.DataFrame([{
        'address':          start_address,
        'interest_score':   0,
        'tour_time':        0,
        'open_house_start': tour_start,
        'open_house_end':   tour_end,
    }])

    end_row = pd.DataFrame([{
        'address':          end_address,
        'interest_score':   0,
        'tour_time':        0,
        'open_house_start': tour_start,
        'open_house_end':   tour_end,
    }])

    return pd.concat([start_row, optw_input_df, end_row], ignore_index=True)

In [11]:
# ── Usage ─────────────────────────────────────────────────────────────────────
start_address = "207 13th St NE, Atlanta, GA 30309"
end_address = "207 13th St NE, Atlanta, GA 30309"
optw_input_df = add_start_and_end(optw_input_df, start_address, end_address)
display(optw_input_df)

,address,interest_score,tour_time,open_house_start,open_house_end
0,"207 13th St NE, Atlanta, GA 30309",0.0,0,09:00,18:00
1,"2964 Olivine Dr SE, Dacula, GA 30019",100.0,30,13:00,15:00
2,"2017 Preserve Creek Way, Loganville, GA 30052",94.5,30,12:00,14:00
3,"2201 Colonial Oak Way, Stone Mountain, GA 30087",91.3,30,11:00,13:00
4,"2583 Madison Mae Ln, Grayson, GA 30017",87.2,30,12:00,15:00
5,"104 Green Love Ln, Grayson, GA 30017",86.8,30,15:00,17:00
6,"67 Wilder Ridge Way, Lawrenceville, GA 30044",84.1,30,10:00,18:00
7,"3109 Misty Springs Path, Lawrenceville, GA 30045",82.2,30,12:00,17:00
8,"207 13th St NE, Atlanta, GA 30309",0.0,0,09:00,18:00


### Geocode Addresses

Three-tier fallback pipeline:
1. **Census geocoder** — free, no API key, fast. May miss newer streets.
2. **Nominatim / OSM** — free, community-maintained. Catches streets Census misses.
3. **Trilateration** — estimates location from school coordinates and Redfin distances when both geocoders fail.

In [12]:
from core.geocoding import (
    geocode_all_homes,
    geocode_summary,
    build_optw_coords
)

# ════════════════════════════════════════════════════════════════════════════
# USAGE
# ════════════════════════════════════════════════════════════════════════════

# ── Step 1: Geocode all homes in optw_input_df ───────────────────────────────
optw_input_df = geocode_all_homes(optw_input_df, df)
geocode_summary(optw_input_df)

# ── Step 2: Build coordinate list (reads depot from optw_input_df rows 0 & -1)
all_coords, start_idx, end_idx = build_optw_coords(optw_input_df)


🔍 [1/8] Trying Census:      2964 Olivine Dr SE, Dacula, GA 30019
   ✅ CENSUS: 33.910296009857, -83.867449311113
🔍 [2/8] Trying Census:      2017 Preserve Creek Way, Loganville, GA 30052
   ✅ CENSUS: 33.836192645351, -83.977136674058
🔍 [3/8] Trying Census:      2201 Colonial Oak Way, Stone Mountain, GA 30087
   ✅ CENSUS: 33.821670984596, -84.104538094621
🔍 [4/8] Trying Census:      2583 Madison Mae Ln, Grayson, GA 30017
   ✅ CENSUS: 33.882335036204, -83.947244018746
🔍 [5/8] Trying Census:      104 Green Love Ln, Grayson, GA 30017
   ⚠️  Census failed. Trying Nominatim: 104 Green Love Ln, Grayson, GA 30017
   ⚠️  Nominatim failed. Trying trilateration: 104 Green Love Ln, Grayson, GA 30017
   ✅ TRILATERATION: 33.891665, -83.964704
🔍 [6/8] Trying Census:      67 Wilder Ridge Way, Lawrenceville, GA 30044
   ⚠️  Census failed. Trying Nominatim: 67 Wilder Ridge Way, Lawrenceville, GA 30044
   ⚠️  Nominatim failed. Trying trilateration: 67 Wilder Ridge Way, Lawrenceville, GA 30044
   ✅ TRILATE

### Build Duration Matrix

Uses OSRM (real road-network routing) with a haversine fallback. The matrix includes arc-blocking for same-depot configurations so the solver can't cheat.

In [14]:
from core.routing import build_duration_matrix_from_coords

# ── Step 3: Build duration matrix ────────────────────────────────────────────
duration_matrix, matrix_method = build_duration_matrix_from_coords(
    all_coords, start_idx, end_idx
)

# Convert to integer matrix for OR-Tools (multiply by 10 to preserve 1 decimal)
duration_matrix_int = [[int(t * 10) for t in row] for row in duration_matrix]

print(f"\n✅ Ready for solver:")
print(f"   Nodes:        {len(all_coords)}")
print(f"   Matrix:       {len(all_coords)}×{len(all_coords)} ({matrix_method})")
print(f"   Same depot:   {all_coords[start_idx] == all_coords[end_idx]}")

✅ Duration matrix built via OSRM (9×9).
ℹ️  Same depot — impossible arcs set to 99999. Matrix stays 9×9.
   Method: osrm | Nodes: 9 | Same depot: True

✅ Ready for solver:
   Nodes:        9
   Matrix:       9×9 (osrm)
   Same depot:   True


# Optimization

With interest scores and a travel time matrix in hand, we solve the OPTW: select and order a subset of homes to maximize total interest score while respecting each home's open house window and a total tour time budget.

## Brute Force (Exact)

Evaluates every possible subset and ordering. Guarantees the optimal solution but only feasible for small N (here N=7 homes → 13,699 routes).

In [15]:
from core.solvers import brute_force_optw

# ── Run ───────────────────────────────────────────────────────────────────────
results = brute_force_optw(optw_input_df, duration_matrix)


── Brute force OPTW results ─────────────────────────────────────────
   Routes evaluated:  13,699
   Feasible routes:   7
   Best score:        626.1
   Total drive time:  139.3 min

── Optimal route ────────────────────────────────────────────────────
   1. 2201 Colonial Oak Way, Stone Mountain, GA 30087
      Arrive: 09:31  (wait 88.7 min)  Visit: 11:00  Depart: 11:30  Score: 91.3
   2. 2017 Preserve Creek Way, Loganville, GA 30052
      Arrive: 11:51  (wait 8.1 min)  Visit: 12:00  Depart: 12:30  Score: 94.5
   3. 2964 Olivine Dr SE, Dacula, GA 30019
      Arrive: 12:57  (wait 3.0 min)  Visit: 13:00  Depart: 13:30  Score: 100.0
   4. 2583 Madison Mae Ln, Grayson, GA 30017
      Arrive: 13:49  Visit: 13:49  Depart: 14:19  Score: 87.2
   5. 104 Green Love Ln, Grayson, GA 30017
      Arrive: 14:23  (wait 36.7 min)  Visit: 15:00  Depart: 15:30  Score: 86.8
   6. 3109 Misty Springs Path, Lawrenceville, GA 30045
      Arrive: 15:45  Visit: 15:45  Depart: 16:15  Score: 82.2
   7. 67 Wilde

## Greedy Heuristic (Baseline)

At each step, selects the highest-scoring feasible home. Fast but suboptimal — included as a baseline to benchmark against.

In [16]:
from core.solvers import greedy_optw

# ── Run ───────────────────────────────────────────────────────────────────────
greedy_results = greedy_optw(optw_input_df, duration_matrix)


── Greedy OPTW results ──────────────────────────────────────────────
   Homes visited:     4/7
   Interest score:    358.1 / 626.1 (57.2% of max)
   Total drive time:  100.5 min
   Solve time:        0.0006s

── Route ────────────────────────────────────────────────────────────
   Start: 207 13th St NE, Atlanta, GA 30309  09:00
   1. 2964 Olivine Dr SE, Dacula, GA 30019
      Arrive: 09:57  (wait 182.9 min)  Visit: 13:00  Depart: 13:30  Score: 100.0
   2. 2583 Madison Mae Ln, Grayson, GA 30017
      Arrive: 13:49  Visit: 13:49  Depart: 14:19  Score: 87.2
   3. 104 Green Love Ln, Grayson, GA 30017
      Arrive: 14:23  (wait 36.7 min)  Visit: 15:00  Depart: 15:30  Score: 86.8
   4. 67 Wilder Ridge Way, Lawrenceville, GA 30044
      Arrive: 15:50  Visit: 15:50  Depart: 16:20  Score: 84.1
   End:   207 13th St NE, Atlanta, GA 30309


# Results

The brute force solver visits all 7 eligible homes (score: 626.1) while the greedy heuristic only manages 4 (score: 358.1, 57% of optimal). The greedy approach wastes ~3 hours waiting at its first stop because it chases the highest-scored home without considering time window alignment — illustrating why routing order matters as much as selection.